# Problem Statement

Find the top 3 customers (by total spending) in each country, along with their total number of transactions and total amount spent.\

Requirements:\
Join:\
sales_transactions\
sales_customers\
Calculate per customer:\
total transactions\
total spend (SUM(totalPrice))\
Rank customers within each country\
Return only top 3 per country\

In [0]:
%sql
select * from samples.bakehouse.sales_customers limit 10;
select * from samples.bakehouse.sales_transactions limit 10;

customerID,first_name,last_name,email_address,phone_number,address,city,state,country,continent,postal_zip_code,gender
2000259,Kayla,Barrett,brittanyramos@example.org,349-683-9514x73065,717 Whitney Roads,Kathrynborough,Massachusetts,Japan,Asia,81587,female
2000260,Amanda,Reed,scollier@example.org,+1-999-308-9110,69075 Logan Circles Apt. 540,East Catherine,Rhode Island,Japan,Asia,6657,female
2000261,Steven,Tanner,haileysanchez@example.net,859-946-4140x24086,08560 Thomas Land,Williamshire,Missouri,Japan,Asia,20642,female
2000262,Jennifer,Forbes,belldonna@example.com,633-427-4977,5840 Warren Garden Suite 901,Delacruzville,Nevada,Australia,Oceania,21440,male
2000263,Kenneth,Berger,bdalton@example.net,(831)220-1833x906,693 Baker Dale,West Wendy,Colorado,Australia,Oceania,32756,female
2000264,James,Edwards,mary39@example.org,271-321-1561x9697,665 Campbell Streets Suite 966,Sherryton,Illinois,Japan,Asia,76717,male
2000265,Ann,Montgomery,michaelreese@example.org,7673757334,220 Barber Islands Apt. 664,East Todd,Iowa,Australia,Oceania,77976,female
2000266,Heather,Moore,mannmartin@example.com,278-606-3676x938,896 Greene Hill Suite 575,Josephburgh,Maryland,Australia,Oceania,2149,male
2000267,Joseph,Graham,suzannereeves@example.org,538.787.9102x4963,2563 Jessica Mountains Apt. 192,Kelseyview,Mississippi,Australia,Oceania,85290,female
2000268,Lisa,Ramirez,kristi04@example.com,(643)361-8721x12062,573 Bailey Lights Suite 941,West Deniseland,Connecticut,Japan,Asia,58198,female


In [0]:
%sql
WITH cte AS (
    SELECT
        c.customerID,
        c.country,
        t.transactionID,
        t.totalPrice
    FROM samples.bakehouse.sales_customers c
    JOIN samples.bakehouse.sales_transactions t
        ON c.customerID = t.customerID
),
customer_agg AS (
    SELECT 
        country,
        customerID,
        SUM(totalPrice) AS total_revenue,
        COUNT(DISTINCT transactionID) AS total_transactions
    FROM cte
    GROUP BY country, customerID
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY country 
            ORDER BY total_revenue DESC
        ) AS rank
    FROM customer_agg
)
SELECT *
FROM ranked
WHERE rank <= 3;

country,customerID,total_revenue,total_transactions,rank
Australia,2000112,672,20,1
Australia,2000248,498,13,2
Australia,2000179,450,11,3
Japan,2000134,519,12,1
Japan,2000006,480,18,2
Japan,2000082,423,14,3
USA,2000275,465,19,1
USA,2000229,450,16,2
USA,2000022,411,18,3


In [0]:
df_customer = spark.read.table("samples.bakehouse.sales_customers")
df_transaction = spark.read.table("samples.bakehouse.sales_transactions")
df_customer.show(10)
df_transaction.show(10)

+----------+----------+----------+--------------------+-------------------+--------------------+---------------+-------------+---------+---------+---------------+------+
|customerID|first_name| last_name|       email_address|       phone_number|             address|           city|        state|  country|continent|postal_zip_code|gender|
+----------+----------+----------+--------------------+-------------------+--------------------+---------------+-------------+---------+---------+---------------+------+
|   2000259|     Kayla|   Barrett|brittanyramos@exa...| 349-683-9514x73065|   717 Whitney Roads| Kathrynborough|Massachusetts|    Japan|     Asia|          81587|female|
|   2000260|    Amanda|      Reed|scollier@example.org|    +1-999-308-9110|69075 Logan Circl...| East Catherine| Rhode Island|    Japan|     Asia|           6657|female|
|   2000261|    Steven|    Tanner|haileysanchez@exa...| 859-946-4140x24086|   08560 Thomas Land|   Williamshire|     Missouri|    Japan|     Asia|    

In [0]:
df = df_customer.join(df_transaction, df_customer.customerID == df_transaction.customerID).select(df_customer.customerID, df_customer.country, df_transaction.transactionID, df_transaction.totalPrice)

df.show(10)


+----------+---------+-------------+----------+
|customerID|  country|transactionID|totalPrice|
+----------+---------+-------------+----------+
|   2000253|    Japan|      1002961|        24|
|   2000226|Australia|      1003007|       108|
|   2000108|    Japan|      1003017|       120|
|   2000173|Australia|      1003068|        84|
|   2000075|      USA|      1003103|        84|
|   2000295|      USA|      1003147|        96|
|   2000237|    Japan|      1003196|       120|
|   2000272|    Japan|      1003329|        84|
|   2000209|      USA|      1001264|        84|
|   2000120|    Japan|      1001287|       120|
+----------+---------+-------------+----------+
only showing top 10 rows


In [0]:
from pyspark.sql.functions import sum, count, row_number
from pyspark.sql.window import Window

df = df.groupBy('customerID', 'country').agg(sum('totalPrice').alias('total_revenue'),count('transactionID').alias('total_transactions'))

window_spec = Window.partitionBy('country').orderBy(col('total_revenue').desc())
df = df.withColumn('rank', row_number().over(window_spec))
df = df.filter(col('rank') <= 3)

display(df)

customerID,country,total_revenue,total_transactions,rank
2000112,Australia,672,20,1
2000248,Australia,498,13,2
2000179,Australia,450,11,3
2000134,Japan,519,12,1
2000006,Japan,480,18,2
2000082,Japan,423,14,3
2000275,USA,465,19,1
2000229,USA,450,16,2
2000022,USA,411,18,3


In [0]:
from pyspark.sql.functions import sum, count, row_number, col
from pyspark.sql.window import Window

window_spec = Window.partitionBy('country').orderBy(col('total_revenue').desc())
df = df.withColumn('rank', row_number().over(window_spec))
df = df.filter(col('rank') <= 3)

display(df)




customerID,country,total_revenue,total_transactions,rank
2000112,Australia,672,20,1
2000248,Australia,498,13,2
2000179,Australia,450,11,3
2000134,Japan,519,12,1
2000006,Japan,480,18,2
2000082,Japan,423,14,3
2000275,USA,465,19,1
2000229,USA,450,16,2
2000022,USA,411,18,3


# Write a Python program that:

Asks the user to enter a number.
Then determines and prints:
Whether the number is positive, negative, or zero
Whether it is even or odd
Whether it is a multiple of 5

In [0]:
number = int(input("Enter a number: "))

if number % 2 == 0:
    print("Even")
else:
    print("Odd")

if number % 5 == 0:
    print("Multiple of 5")
else:
    print("Not multiple of 5")

if number > 0:
    print("Positive")
elif number == 0:
    print("Zero")
else:
    print("Negative")

Enter a number:  10

Even
Multiple of 5
Positive
